# 01 — Departure Data Preparation (V9.0)

**加载、清洗 Jan-Oct 2025 出发航班数据 + 到达数据 + 天气 + FAA**

In [1]:
# 0. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_curve
)
from sklearn.isotonic import IsotonicRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import catboost as cb
import lightgbm as lgb
import shap
import joblib
import json
import os
import sys

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
if not (BASE_DIR / 'src').exists():
    BASE_DIR = Path('../../..')

DATA_RAW = BASE_DIR / 'data' / 'raw' / 'LGA_Dataset'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Add src to path
sys.path.insert(0, str(BASE_DIR / 'src'))

from features.lag_features import add_lag_features, add_congestion_features, compute_v4_lag_features
from features.aircraft_features import compute_prev_aircraft_delay
from models.calibration import (
    fit_isotonic_calibration, apply_calibration, compute_ece
)
from models.temporal_weights import compute_temporal_weights, combine_temporal_and_class_weights

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


## 1. Departure Flight Data

In [2]:
# 1.1 Load departure CSVs (5 files, Jan-Oct 2025)
# Reuse harmonize_flight_columns() from NB01 pattern (3-schema: old/v1/v2)

# === Carrier code → Marketing Airline name mapping ===
CARRIER_TO_AIRLINE = {
    # --- Mainline ---
    'AAL': 'American Airlines',
    'DAL': 'Delta Air Lines',
    'UAL': 'United Airlines',
    'SWA': 'Southwest Airlines',
    'FFT': 'Frontier Airlines',
    'JBU': 'JetBlue Airways',
    'NKS': 'Spirit Airlines',
    'ACA': 'Air Canada',
    'POE': 'Porter Airlines',
    'GXA': 'GlobalX Airlines',
    # --- Regional (operating as mainline brand at LGA) ---
    'RPA': 'American Airlines',   # Republic Airways → American Eagle
    'ASH': 'American Airlines',   # Mesa Airlines → American Eagle
    'EDV': 'Delta Air Lines',     # Endeavor Air → Delta Connection
    'SKW': 'United Airlines',     # SkyWest → United Express
    'GJS': 'United Airlines',     # GoJet → United Express
    'JZA': 'Air Canada',          # Jazz Aviation → Air Canada Express
    # --- General Aviation / Charter ---
    'EJA': 'General Aviation', 'LXJ': 'General Aviation', 'EJM': 'General Aviation',
    'GPD': 'General Aviation', 'CNS': 'General Aviation', 'VJA': 'General Aviation',
    'TWY': 'General Aviation', 'ASP': 'General Aviation', 'BOG': 'General Aviation',
    'COL': 'General Aviation', 'WUP': 'General Aviation', 'JRE': 'General Aviation',
    'XFL': 'General Aviation',
}

# Column mapping: v2 schema → canonical (old) schema
NEW_V2_TO_CANONICAL = {
    'Body Type': 'Body Type Desc',
    'Delay': 'Total Calculated Delay',
    'Movement Time': 'Total  Movement Time',       # old has double space
    'Ramp Time': 'Total Ramp Time',
    'total_taxi_time': 'Total Taxi Time Calc',      # lowercase in v2!
    'Gate Occupancy Time': 'Total Gate Occ Time',
}

# Column mapping: v1 schema → canonical
NEW_V1_TO_CANONICAL = {
    'Airline Name': 'Marketing Airline Desc',
    'Body Type': 'Body Type Desc',
    'Delay': 'Total Calculated Delay',
    'Movement Time': 'Total  Movement Time',
    'Taxi Time': 'Total Taxi Time Calc',
}

def harmonize_flight_columns(df, direction='arrival'):
    """Detect schema variant and rename to canonical (old) column names.
    Handles 3 schemas: Old (May-Sept), New v1 (Airline Name), New v2 (Carrier code)."""
    if 'Carrier' in df.columns:
        # === New v2 schema (Jan-May, Sept-Oct full data) ===
        rename_map = NEW_V2_TO_CANONICAL.copy()
        if 'Runway' in df.columns:
            rename_map['Runway'] = 'Arrival Runway' if direction == 'arrival' else 'Departure Runway'
        df = df.rename(columns=rename_map)
        df['Marketing Airline Desc'] = df['Carrier'].map(CARRIER_TO_AIRLINE).fillna('General Aviation')
        if 'Body Type Desc' in df.columns:
            df['Body Type Desc'] = df['Body Type Desc'].replace('NULL', np.nan)
        if 'Terminal' in df.columns and 'Terminal Code' in df.columns:
            df = df.drop(columns=['Terminal'])
        n_mapped = df['Marketing Airline Desc'].ne('General Aviation').sum()
        n_ga = (df['Marketing Airline Desc'] == 'General Aviation').sum()
        print(f"  → v2 schema: {n_mapped:,} commercial, {n_ga:,} GA/other")
    elif 'Airline Name' in df.columns:
        # === New v1 schema ===
        rename_map = NEW_V1_TO_CANONICAL.copy()
        if 'Runway' in df.columns:
            rename_map['Runway'] = 'Arrival Runway' if direction == 'arrival' else 'Departure Runway'
        df = df.rename(columns=rename_map)
        if 'Body Type Desc' in df.columns:
            df['Body Type Desc'] = df['Body Type Desc'].replace('NULL', np.nan)
        if 'Terminal' in df.columns and 'Terminal Code' in df.columns:
            df = df.drop(columns=['Terminal'])
        print(f"  → v1 schema: harmonized")
    else:
        # === Old schema (May-Sept) ===
        print(f"  → Old schema, no renaming needed")
    return df

dep_files = [
    DATA_RAW / 'LGA_Jan-May2025_Departures.csv',       # v2 schema
    DATA_RAW / 'LGA_May-June2025_Departures.csv',       # old schema
    DATA_RAW / 'LGA_July2025_Departures.csv',           # old schema
    DATA_RAW / 'LGA_Aug-Sept2025_Departures.csv',       # old schema
    DATA_RAW / 'LGA_Sept-Oct2025_Departures.csv',       # v2 schema
]
dfs = []
for f in dep_files:
    if f.exists():
        tmp = pd.read_csv(f)
        print(f'{f.name}: {len(tmp):,} rows')
        tmp = harmonize_flight_columns(tmp, direction='departure')
        dfs.append(tmp)
    else:
        print(f'WARNING: {f.name} not found — skipping')

df_dep = pd.concat(dfs, ignore_index=True)

# Sort and dedup
df_dep['_sort_dt'] = pd.to_datetime(df_dep['Block Schedule'], errors='coerce')
df_dep = df_dep.sort_values('_sort_dt').drop(columns=['_sort_dt'])
before = len(df_dep)
df_dep = df_dep.drop_duplicates(
    subset=['Registration', 'Call Sign', 'Block Schedule', 'Block Actual'],
    keep='first'
).reset_index(drop=True)
print(f'\nDedup: {before} → {len(df_dep)} ({before - len(df_dep)} removed)')
print(f'Total departures: {len(df_dep):,}')

LGA_Jan-May2025_Departures.csv: 67,605 rows
  → v2 schema: 66,550 commercial, 1,055 GA/other
LGA_May-June2025_Departures.csv: 20,192 rows
  → Old schema, no renaming needed
LGA_July2025_Departures.csv: 15,960 rows
  → Old schema, no renaming needed
LGA_Aug-Sept2025_Departures.csv: 16,303 rows
  → Old schema, no renaming needed
LGA_Sept-Oct2025_Departures.csv: 30,044 rows
  → v2 schema: 29,547 commercial, 497 GA/other

Dedup: 150104 → 148061 (2043 removed)
Total departures: 148,061


## 2. DateTime & Target Variable

In [3]:
# 1.2 Parse datetime & create target

# Parse scheduled departure datetime
# Try Block Schedule first, fallback to Off Scheduled
df_dep['_block_dt'] = pd.to_datetime(df_dep['Block Schedule'], format='mixed', errors='coerce')
df_dep['_off_dt'] = pd.to_datetime(df_dep['Off Scheduled'], format='mixed', errors='coerce')
df_dep['Scheduled_Departure_Datetime'] = df_dep['_block_dt'].fillna(df_dep['_off_dt'])
df_dep.drop(columns=['_block_dt', '_off_dt'], inplace=True)

# Drop rows without valid datetime
n_before = len(df_dep)
df_dep = df_dep.dropna(subset=['Scheduled_Departure_Datetime'])
print(f'Dropped {n_before - len(df_dep)} rows without valid datetime')

# Extract date, hour
df_dep['Date'] = df_dep['Scheduled_Departure_Datetime'].dt.date
df_dep['Date'] = pd.to_datetime(df_dep['Date'])
df_dep['Hour'] = df_dep['Scheduled_Departure_Datetime'].dt.hour
df_dep['Month'] = df_dep['Scheduled_Departure_Datetime'].dt.month

# Parse delay
df_dep['Dep_Calculated_Delay'] = pd.to_numeric(
    df_dep['Total Calculated Delay'], errors='coerce'
)

# Drop rows without delay
n_before = len(df_dep)
df_dep = df_dep.dropna(subset=['Dep_Calculated_Delay'])
print(f'Dropped {n_before - len(df_dep)} rows without delay value')

# Target variable (DOT standard: >15 min)
df_dep['Is_Delayed'] = (df_dep['Dep_Calculated_Delay'] > 15).astype(int)

# Clean terminal
df_dep['Terminal_Clean'] = df_dep['Terminal Code'].fillna('Unknown').str.strip()

# Clean runway
df_dep['Departure_Runway_Clean'] = df_dep['Departure Runway'].fillna('Unknown').astype(str).str.strip()

# Parse taxi time
df_dep['Dep_Taxi_Time'] = pd.to_numeric(df_dep['Total Taxi Time Calc'], errors='coerce')

print(f'\nFinal departure dataset: {len(df_dep):,} flights')
print(f'Delayed rate: {df_dep["Is_Delayed"].mean():.1%}')
print(f'Date range: {df_dep["Date"].min()} to {df_dep["Date"].max()}')
print(f'\nDelay distribution (minutes):')
print(df_dep['Dep_Calculated_Delay'].describe())

Dropped 582 rows without valid datetime
Dropped 0 rows without delay value

Final departure dataset: 147,479 flights
Delayed rate: 20.3%
Date range: 2024-12-31 00:00:00 to 2025-10-31 00:00:00

Delay distribution (minutes):
count    147479.000000
mean         13.301975
std          57.750622
min         -94.000000
25%          -7.000000
50%          -4.000000
75%           8.000000
max        2590.000000
Name: Dep_Calculated_Delay, dtype: float64


## 3. Arrival Data

In [4]:
# 1.3 Load arrival data (5 files, for aircraft continuity + lga_arr_delay_1h)

arr_files = [
    DATA_RAW / 'LGA_Jan-May2025_Arrivals.csv',       # new schema
    DATA_RAW / 'LGA_May-June2025_Arrivals.csv',       # old schema
    DATA_RAW / 'LGA_July2025_Arrivals.csv',           # old schema
    DATA_RAW / 'LGA_Aug-Sept2025_Arrivals.csv',       # old schema
    DATA_RAW / 'LGA_Sept-Oct2025_Arrivals.csv',       # new schema
]
arr_dfs = []
for f in arr_files:
    if f.exists():
        tmp = pd.read_csv(f)
        print(f'{f.name}: {len(tmp):,} rows')
        tmp = harmonize_flight_columns(tmp, direction='arrival')
        arr_dfs.append(tmp)
    else:
        print(f'WARNING: {f.name} not found — skipping')

df_arr = pd.concat(arr_dfs, ignore_index=True)

# Parse arrival datetime
df_arr['_block_dt'] = pd.to_datetime(df_arr['Block Schedule'], format='mixed', errors='coerce')
df_arr['_on_dt'] = pd.to_datetime(df_arr.get('On Scheduled'), format='mixed', errors='coerce') if 'On Scheduled' in df_arr.columns else pd.NaT
df_arr['Scheduled_Arrival_Datetime'] = df_arr['_block_dt'].fillna(df_arr['_on_dt'])
df_arr.drop(columns=['_block_dt', '_on_dt'], inplace=True, errors='ignore')
df_arr = df_arr.dropna(subset=['Scheduled_Arrival_Datetime'])

df_arr['Date'] = pd.to_datetime(df_arr['Scheduled_Arrival_Datetime'].dt.date)
df_arr['Arr_Calculated_Delay'] = pd.to_numeric(
    df_arr['Total Calculated Delay'], errors='coerce'
)
df_arr = df_arr.dropna(subset=['Arr_Calculated_Delay'])

# Dedup
before = len(df_arr)
df_arr = df_arr.drop_duplicates(
    subset=['Registration', 'Call Sign', 'Block Schedule', 'Block Actual'],
    keep='first'
).reset_index(drop=True)

print(f'\nArrivals loaded: {len(df_arr):,} flights (dedup: {before}→{len(df_arr)})')
print(f'Date range: {df_arr["Date"].min()} to {df_arr["Date"].max()}')

LGA_Jan-May2025_Arrivals.csv: 67,997 rows
  → v2 schema: 66,643 commercial, 1,354 GA/other
LGA_May-June2025_Arrivals.csv: 20,244 rows
  → Old schema, no renaming needed
LGA_July2025_Arrivals.csv: 16,039 rows
  → Old schema, no renaming needed
LGA_Aug-Sept2025_Arrivals.csv: 16,415 rows
  → Old schema, no renaming needed
LGA_Sept-Oct2025_Arrivals.csv: 30,141 rows
  → v2 schema: 29,486 commercial, 655 GA/other

Arrivals loaded: 147,168 flights (dedup: 147578→147168)
Date range: 2024-12-31 00:00:00 to 2025-10-31 00:00:00


## 4. LGA Weather

In [5]:
# 1.4 Load LGA weather (3 sources — old aggregated + 2 new WeatherFull)

# --- Old format (May-Sept, pre-aggregated hourly) ---
lga_wx_parts = []

lga_wx_path = DATA_RAW / 'LGA_Summer2025_Weather.csv'
if lga_wx_path.exists():
    df_lga_wx = pd.read_csv(lga_wx_path)
    print(f'LGA weather (old format): {len(df_lga_wx)} rows')
    
    df_lga_wx['_date'] = pd.to_datetime(df_lga_wx['Date']).dt.date
    df_lga_wx['_time'] = pd.to_datetime(df_lga_wx['Military Time']).dt.time
    df_lga_wx['_dt'] = pd.to_datetime(
        df_lga_wx['_date'].astype(str) + ' ' + df_lga_wx['_time'].astype(str), errors='coerce'
    )
    df_lga_wx = df_lga_wx.dropna(subset=['_dt'])
    df_lga_wx['_hour'] = df_lga_wx['_dt'].dt.floor('h')
    
    desc_col = [c for c in df_lga_wx.columns if 'weather' in c.lower() and 'desc' in c.lower()]
    if desc_col:
        df_lga_wx['_storm'] = df_lga_wx[desc_col[0]].str.contains(
            'Thunder|Storm|T-Storm', case=False, na=False).astype(int)
    else:
        df_lga_wx['_storm'] = 0
    
    precip_col = [c for c in df_lga_wx.columns if 'precip' in c.lower()]
    wind_col = [c for c in df_lga_wx.columns if 'wind' in c.lower() and 'gust' in c.lower()]
    vis_col = [c for c in df_lga_wx.columns if 'visib' in c.lower()]
    
    agg_dict = {'_storm': 'max'}
    rename_dict = {'_storm': 'lga_storm_flag'}
    if precip_col:
        df_lga_wx['_precip'] = pd.to_numeric(df_lga_wx[precip_col[0]], errors='coerce').fillna(0)
        agg_dict['_precip'] = 'sum'; rename_dict['_precip'] = 'lga_precip'
    if wind_col:
        df_lga_wx['_wind'] = pd.to_numeric(df_lga_wx[wind_col[0]], errors='coerce').fillna(0)
        agg_dict['_wind'] = 'max'; rename_dict['_wind'] = 'lga_wind_gust'
    if vis_col:
        df_lga_wx['_vis'] = pd.to_numeric(df_lga_wx[vis_col[0]], errors='coerce').fillna(10)
        agg_dict['_vis'] = 'min'; rename_dict['_vis'] = 'lga_visibility'
    
    old_hourly = df_lga_wx.groupby('_hour').agg(agg_dict).reset_index()
    old_hourly = old_hourly.rename(columns=rename_dict).rename(columns={'_hour': 'wx_hour'})
    lga_wx_parts.append(old_hourly)
    print(f'  → {len(old_hourly)} hourly records ({old_hourly["wx_hour"].min()} to {old_hourly["wx_hour"].max()})')

# --- New WeatherFull format (Jan-May, Sept-Oct) ---
weather_full_files = [
    DATA_RAW / 'LGA_Jan-May2025_WeatherFull.csv',
    DATA_RAW / 'LGA_Sept-Oct2025_WeatherFull.csv',
]
for wf_path in weather_full_files:
    if wf_path.exists():
        wf = pd.read_csv(wf_path)
        if 'key' in wf.columns:
            wf = wf[wf['key'] == 'KLGA']
        print(f'LGA weather (new format): {wf_path.name} — {len(wf)} rows')
        
        dt = pd.to_datetime(wf['valid_time_est'], format='mixed', errors='coerce')
        if dt.dt.tz is not None:
            dt = dt.dt.tz_localize(None)
        wf['_hour'] = dt.dt.floor('h')
        
        wf['_storm'] = wf['wx_phrase'].str.contains('Thunder|Storm|T-Storm', case=False, na=False).astype(int)
        for col in ['precip_hrly', 'gust', 'vis']:
            wf[col] = pd.to_numeric(wf[col], errors='coerce')
        
        new_hourly = wf.groupby('_hour').agg({
            'precip_hrly': 'sum', 'gust': 'max', 'vis': 'min', '_storm': 'max'
        }).reset_index()
        new_hourly.columns = ['wx_hour', 'lga_precip', 'lga_wind_gust', 'lga_visibility', 'lga_storm_flag']
        lga_wx_parts.append(new_hourly)
        print(f'  → {len(new_hourly)} hourly records ({new_hourly["wx_hour"].min()} to {new_hourly["wx_hour"].max()})')
    else:
        print(f'NOT FOUND: {wf_path.name}')

# --- Combine and deduplicate ---
if lga_wx_parts:
    lga_wx_hourly = pd.concat(lga_wx_parts, ignore_index=True)
    lga_wx_hourly = lga_wx_hourly.sort_values('wx_hour').drop_duplicates(subset='wx_hour', keep='first').reset_index(drop=True)
    print(f'\nUnified LGA weather: {len(lga_wx_hourly)} hourly records')
    print(f'  Date range: {lga_wx_hourly["wx_hour"].min()} to {lga_wx_hourly["wx_hour"].max()}')
    print(f'  Storm flag mean: {lga_wx_hourly["lga_storm_flag"].mean():.3f}')
    HAS_LGA_WX = True
else:
    lga_wx_hourly = None
    HAS_LGA_WX = False
    print('WARNING: No LGA weather data available')

LGA weather (old format): 2800 rows
  → 2471 hourly records (2025-05-22 00:00:00 to 2025-09-01 23:00:00)
LGA weather (new format): LGA_Jan-May2025_WeatherFull.csv — 3963 rows
  → 3362 hourly records (2025-01-01 00:00:00 to 2025-05-21 23:00:00)
LGA weather (new format): LGA_Sept-Oct2025_WeatherFull.csv — 1623 rows
  → 1463 hourly records (2025-09-01 00:00:00 to 2025-10-31 23:00:00)

Unified LGA weather: 7272 hourly records
  Date range: 2025-01-01 00:00:00 to 2025-10-31 23:00:00
  Storm flag mean: 0.005


## 5. Destination Weather

In [6]:
# 1.5 Load destination weather (Non-PA Airport Weather)
# Expanded to 4 files (Jan-Oct 2025)
# TIMEZONE: use valid_time_est (not valid_time_gmt!) — see CLAUDE.md 禁忌 #11

dest_wx_files = [
    DATA_RAW / 'LGA_NonPAAirportWeather_Jan-May2025.csv',
    DATA_RAW / 'LGA_NonPAAirportWeather_May-June2025.csv',
    DATA_RAW / 'LGA_NonPAAirportWeather_July-Sept2025.csv',
    DATA_RAW / 'LGA_NonPAAirportWeather_Sept-Oct2025.csv',
]

dest_wx_dfs = []
for f in dest_wx_files:
    if f.exists():
        dest_wx_dfs.append(pd.read_csv(f))
        print(f'  Loaded {f.name}: {len(dest_wx_dfs[-1]):,} rows')
    else:
        print(f'  NOT FOUND: {f.name}')

if dest_wx_dfs:
    df_dest_wx = pd.concat(dest_wx_dfs, ignore_index=True)
    print(f'\nDestination weather loaded: {len(df_dest_wx):,} rows')
    print(f'Columns: {df_dest_wx.columns.tolist()[:15]}...')
    
    # Parse datetime — use valid_time_est (EST values with wrong Z suffix), strip tz
    dt = pd.to_datetime(df_dest_wx['valid_time_est'], format='mixed', errors='coerce')
    if dt.dt.tz is not None:
        dt = dt.dt.tz_localize(None)
    df_dest_wx['_valid_dt'] = dt
    df_dest_wx = df_dest_wx.dropna(subset=['_valid_dt'])
    df_dest_wx['_wx_hour'] = df_dest_wx['_valid_dt'].dt.floor('h')
    print(f'  After datetime parse: {len(df_dest_wx):,} rows')
    
    # Airport code — obs_id is lowercase ICAO (e.g. "kjfk") → uppercase
    if 'obs_id' in df_dest_wx.columns:
        df_dest_wx['_airport'] = df_dest_wx['obs_id'].str.upper().str.strip()
    elif 'station' in df_dest_wx.columns:
        df_dest_wx['_airport'] = df_dest_wx['station'].str.upper().str.strip()
    else:
        df_dest_wx['_airport'] = df_dest_wx.iloc[:, 0].astype(str).str.upper().str.strip()
    
    print(f'  Sample airport codes: {df_dest_wx["_airport"].unique()[:5]}')
    print(f'  Unique airports: {df_dest_wx["_airport"].nunique()}')
    
    # Numeric features — use NB02-verified column names
    df_dest_wx['_precip'] = pd.to_numeric(df_dest_wx.get('precip_hrly', pd.Series(dtype=float)), errors='coerce').fillna(0)
    df_dest_wx['_dewpt'] = pd.to_numeric(df_dest_wx.get('dewPt', pd.Series(dtype=float)), errors='coerce')
    df_dest_wx['_temp'] = pd.to_numeric(df_dest_wx.get('temp', pd.Series(dtype=float)), errors='coerce')
    df_dest_wx['_pressure'] = pd.to_numeric(df_dest_wx.get('pressure', pd.Series(dtype=float)), errors='coerce')
    
    # Storm flag from wx_phrase
    if 'wx_phrase' in df_dest_wx.columns:
        df_dest_wx['_storm'] = df_dest_wx['wx_phrase'].str.contains(
            'Thunder|Storm|T-Storm', case=False, na=False
        ).astype(int)
    else:
        df_dest_wx['_storm'] = 0
    
    # Cloud cover from clds
    cloud_map = {'CLR': 0, 'FEW': 1, 'SCT': 2, 'BKN': 3, 'OVC': 4}
    if 'clds' in df_dest_wx.columns:
        df_dest_wx['_cloud'] = df_dest_wx['clds'].str.upper().str.strip().map(cloud_map).fillna(2)
    else:
        df_dest_wx['_cloud'] = 2
    
    # Aggregate to hourly per airport
    dest_wx_hourly = df_dest_wx.groupby(['_airport', '_wx_hour']).agg({
        '_precip': 'sum',
        '_dewpt': 'mean',
        '_temp': 'mean',
        '_pressure': 'mean',
        '_storm': 'max',
        '_cloud': 'max',
    }).reset_index()
    dest_wx_hourly.columns = [
        'dest_airport', 'wx_hour',
        'dest_precip', 'dest_dewpoint', 'dest_temp',
        'dest_pressure', 'dest_storm_flag', 'dest_cloud_cover'
    ]
    
    # Pressure change 3h
    dest_wx_hourly = dest_wx_hourly.sort_values(['dest_airport', 'wx_hour'])
    dest_wx_hourly['dest_pressure_change_3h'] = dest_wx_hourly.groupby('dest_airport')['dest_pressure'].diff(3)
    
    print(f'Destination weather hourly: {len(dest_wx_hourly):,} rows, '
          f'{dest_wx_hourly["dest_airport"].nunique()} airports')
    print(f'  Sample dest_airport values: {dest_wx_hourly["dest_airport"].unique()[:5]}')
    HAS_DEST_WX = len(dest_wx_hourly) > 0
else:
    print('Destination weather files not found — will skip destination weather features')
    dest_wx_hourly = None
    HAS_DEST_WX = False

  Loaded LGA_NonPAAirportWeather_Jan-May2025.csv: 1,324,881 rows
  Loaded LGA_NonPAAirportWeather_May-June2025.csv: 354,979 rows
  Loaded LGA_NonPAAirportWeather_July-Sept2025.csv: 629,331 rows
  Loaded LGA_NonPAAirportWeather_Sept-Oct2025.csv: 646,520 rows

Destination weather loaded: 2,955,711 rows
Columns: ['expire_time_est', 'valid_time_est', 'valid_time_gmt', 'key', 'class', 'obs_id', 'obs_name', 'day_ind', 'temp', 'wx_icon', 'icon_extd', 'wx_phrase', 'pressure_tend', 'pressure_desc', 'dewPt']...
  After datetime parse: 2,955,711 rows
  Sample airport codes: ['KTTA' 'KMQI' 'KAUH' 'KMQJ' 'KPBX']
  Unique airports: 606
Destination weather hourly: 1,980,107 rows, 606 airports
  Sample dest_airport values: ['ABE' 'ACK' 'ACY' 'ADS' 'ADW']


## 6. FAA Operational Data

In [7]:
# 1.6 Load FAA data (all 4 types, expanded)

def strip_faa_metadata(df_faa):
    """Remove metadata rows at end of FAA CSVs."""
    if df_faa is None or len(df_faa) == 0:
        return df_faa
    first_col = df_faa.columns[0]
    mask = df_faa[first_col].astype(str).str.contains(r'\d{1,2}/\d{1,2}/\d{4}', na=False)
    removed = (~mask).sum()
    if removed > 0:
        print(f'    Stripped {removed} metadata rows')
    return df_faa[mask].reset_index(drop=True)

def load_faa_file(data_dir, name_candidates, label):
    for fname in name_candidates:
        fpath = data_dir / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            print(f'  {label}: {fname} — {len(df)} rows (raw)')
            return df
    print(f'  {label}: NOT FOUND')
    return None

print('Loading FAA data (expanded)...')

df_gdp = strip_faa_metadata(load_faa_file(DATA_RAW, [
    'LGA_FAA_GroundDelays2025.csv', 'LGA_FAA_GrounDelays2025.csv'], 'GDP'))
df_gsp = strip_faa_metadata(load_faa_file(DATA_RAW, [
    'LGA_FAA_GroundStops2025.csv'], 'GSP'))
df_dd = strip_faa_metadata(load_faa_file(DATA_RAW, [
    'LGA_FAA_DepartureDelays2025.csv'], 'DD'))
df_ad = strip_faa_metadata(load_faa_file(DATA_RAW, [
    'LGA_FAA_ArrivalDelays2025.csv'], 'AD'))

# Harmonize DD column name
if df_dd is not None and 'DD Reason Description' in df_dd.columns:
    df_dd = df_dd.rename(columns={'DD Reason Description': 'DD Reason'})

# Parse datetimes
if df_gdp is not None:
    df_gdp['GDP_Start'] = pd.to_datetime(df_gdp['GDP Start Time'], format='mixed', errors='coerce')
    df_gdp['GDP_End'] = pd.to_datetime(df_gdp['GDP End Time'], format='mixed', errors='coerce')
    df_gdp['GDP_Cancelled'] = pd.to_datetime(df_gdp.get('GDP Cancelled Time'), errors='coerce')
    df_gdp['GDP_Effective_End'] = df_gdp['GDP_Cancelled'].fillna(df_gdp['GDP_End'])
    df_gdp['GDP_Avg_Delay_Min'] = pd.to_numeric(df_gdp['GDP Avg Delay'], errors='coerce')
if df_gsp is not None:
    df_gsp['GSP_Start'] = pd.to_datetime(df_gsp['GSP Start Time'], format='mixed', errors='coerce')
    df_gsp['GSP_End'] = pd.to_datetime(df_gsp['GSP End Time'], format='mixed', errors='coerce')
    df_gsp['GSP_Cancelled'] = pd.to_datetime(df_gsp.get('GSP Cancelled Time'), errors='coerce')
    df_gsp['GSP_Effective_End'] = df_gsp['GSP_Cancelled'].fillna(df_gsp['GSP_End'])
if df_dd is not None:
    df_dd['DD_Update'] = pd.to_datetime(df_dd['DD Update Time'], format='mixed', errors='coerce')
    df_dd['DD_Avg_Delay_Min'] = pd.to_numeric(df_dd['DD Avg Delay'], errors='coerce')
if df_ad is not None:
    df_ad['AD_Update'] = pd.to_datetime(df_ad['AD Update Time'], format='mixed', errors='coerce')

faa_counts = {k: len(v) if v is not None else 0 for k, v in 
              [('GDP', df_gdp), ('GSP', df_gsp), ('DD', df_dd), ('AD', df_ad)]}
print(f'\nFAA summary: {faa_counts}, total={sum(faa_counts.values())}')
HAS_FAA = any(v > 0 for v in faa_counts.values())

Loading FAA data (expanded)...
  GDP: LGA_FAA_GrounDelays2025.csv — 69 rows (raw)
    Stripped 2 metadata rows
  GSP: LGA_FAA_GroundStops2025.csv — 273 rows (raw)
    Stripped 2 metadata rows
  DD: LGA_FAA_DepartureDelays2025.csv — 694 rows (raw)
    Stripped 2 metadata rows
  AD: LGA_FAA_ArrivalDelays2025.csv — 57 rows (raw)
    Stripped 2 metadata rows

FAA summary: {'GDP': 67, 'GSP': 271, 'DD': 692, 'AD': 55}, total=1085


## 7. EDA

In [8]:
# 1.7 EDA: Departure delay distribution

print('=== Departure Delay Distribution ===')
print(f'Total flights: {len(df_dep):,}')
print(f'Delayed (>15 min): {df_dep["Is_Delayed"].sum():,} ({df_dep["Is_Delayed"].mean():.1%})')
print(f'On-time or early: {(~df_dep["Is_Delayed"].astype(bool)).sum():,}')
print(f'\nDelay stats (minutes): mean={df_dep["Dep_Calculated_Delay"].mean():.1f}, '
      f'median={df_dep["Dep_Calculated_Delay"].median():.1f}, '
      f'std={df_dep["Dep_Calculated_Delay"].std():.1f}')

print(f'\n--- By Terminal ---')
for t, g in df_dep.groupby('Terminal_Clean'):
    print(f'  {t}: {len(g):,} flights, {g["Is_Delayed"].mean():.1%} delayed')

print(f'\n--- By Month ---')
for m, g in df_dep.groupby('Month'):
    print(f'  Month {m}: {len(g):,} flights, {g["Is_Delayed"].mean():.1%} delayed')

print(f'\n--- By Hour (top 5 busiest) ---')
hour_stats = df_dep.groupby('Hour').agg(
    flights=('Is_Delayed', 'count'),
    delay_rate=('Is_Delayed', 'mean')
).sort_values('flights', ascending=False)
print(hour_stats.head(5).to_string())

=== Departure Delay Distribution ===
Total flights: 147,479
Delayed (>15 min): 29,925 (20.3%)
On-time or early: 117,554

Delay stats (minutes): mean=13.3, median=-4.0, std=57.8

--- By Terminal ---
  LGA-Unknown: 2,373 flights, 2.6% delayed
  Terminal A: 4,833 flights, 15.7% delayed
  Terminal B: 70,410 flights, 20.9% delayed
  Terminal C: 69,863 flights, 20.6% delayed

--- By Month ---
  Month 1: 13,890 flights, 16.6% delayed
  Month 2: 12,999 flights, 17.6% delayed
  Month 3: 14,839 flights, 17.6% delayed
  Month 4: 15,082 flights, 15.7% delayed
  Month 5: 15,646 flights, 23.7% delayed
  Month 6: 14,526 flights, 28.8% delayed
  Month 7: 14,476 flights, 29.2% delayed
  Month 8: 15,433 flights, 18.2% delayed
  Month 9: 15,159 flights, 14.2% delayed
  Month 10: 15,427 flights, 21.3% delayed
  Month 12: 2 flights, 100.0% delayed

--- By Hour (top 5 busiest) ---
      flights  delay_rate
Hour                     
15      11409    0.266456
8       10731    0.094586
6       10127    0.05964

## 8. Train/Test Split

In [9]:
# 1.8 Time-based Train/Test Split (70/30)

df_dep['Date'] = pd.to_datetime(df_dep['Date'])
cutoff = df_dep['Date'].quantile(0.7)
print(f'Split cutoff: {cutoff}')

train = df_dep[df_dep['Date'] <= cutoff].copy()
test = df_dep[df_dep['Date'] > cutoff].copy()

print(f'Train: {len(train):,} flights ({train["Date"].min()} to {train["Date"].max()}), '
      f'delay rate={train["Is_Delayed"].mean():.1%}')
print(f'Test:  {len(test):,} flights ({test["Date"].min()} to {test["Date"].max()}), '
      f'delay rate={test["Is_Delayed"].mean():.1%}')

# Split arrival data too (for aircraft continuity cross-join)
arr_train = df_arr[df_arr['Date'] <= cutoff].copy()
arr_test_ctx = df_arr[df_arr['Date'] > (cutoff - pd.Timedelta(days=1))].copy()

Split cutoff: 2025-08-04 00:00:00
Train: 103,366 flights (2024-12-31 00:00:00 to 2025-08-04 00:00:00), delay rate=21.3%
Test:  44,113 flights (2025-08-05 00:00:00 to 2025-10-31 00:00:00), delay rate=17.8%


## 9. Save

In [10]:
# === Save intermediate data for downstream notebooks ===
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

def safe_to_parquet(df, path):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str)
    df.to_parquet(path, index=False)

safe_to_parquet(train, DATA_PROCESSED / 'dep_train_raw.parquet')
safe_to_parquet(test, DATA_PROCESSED / 'dep_test_raw.parquet')
safe_to_parquet(df_arr, DATA_PROCESSED / 'dep_arrivals_for_continuity.parquet')
safe_to_parquet(arr_train, DATA_PROCESSED / 'dep_arr_train.parquet')
safe_to_parquet(arr_test_ctx, DATA_PROCESSED / 'dep_arr_test_ctx.parquet')

import pickle
dep_context = {
    'cutoff': cutoff,
    'df_lga_wx': df_lga_wx if 'df_lga_wx' in dir() else None,
    'df_dest_wx': df_dest_wx if 'df_dest_wx' in dir() else None,
    'lga_wx_hourly': lga_wx_hourly if 'lga_wx_hourly' in dir() else None,
    'dest_wx_hourly': dest_wx_hourly if 'dest_wx_hourly' in dir() else None,
    'df_gdp': df_gdp if 'df_gdp' in dir() else None,
    'df_gsp': df_gsp if 'df_gsp' in dir() else None,
    'df_dd': df_dd if 'df_dd' in dir() else None,
    'df_ad': df_ad if 'df_ad' in dir() else None,
    'HAS_FAA': HAS_FAA if 'HAS_FAA' in dir() else False,
    'HAS_LGA_WX': HAS_LGA_WX if 'HAS_LGA_WX' in dir() else False,
    'HAS_DEST_WX': HAS_DEST_WX if 'HAS_DEST_WX' in dir() else False,
}
pickle.dump(dep_context, open(DATA_PROCESSED / 'dep_context.pkl', 'wb'))
print(f'Saved: train={len(train)}, test={len(test)}, arrivals={len(df_arr)}')
print(f'Flags: FAA={dep_context["HAS_FAA"]}, LGA_WX={dep_context["HAS_LGA_WX"]}, DEST_WX={dep_context["HAS_DEST_WX"]}')


Saved: train=103366, test=44113, arrivals=147168
Flags: FAA=True, LGA_WX=True, DEST_WX=True
